In [ ]:
using Bessels, MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
ω_a = 3.8533*2*pi#(rand(1)[1]/4+0.875)

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.5127272727272727
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0;#3e-3*κ_2 #from Leon's paper

# Large sweep

In [ ]:
ϵ_0_list = range(start = 0.11, stop = 1, length = 10);


t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(sqrt(2)*ω_p)*2
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,25)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            a_rot = a.*exp.(1im*ω_a*t_list)

            Final_mode[(re_i,im_j)] = a_rot
        end
    end
    Final_all[ϵ_0] = Final_mode
end

In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(-15,15),
          ylims=(-6,6),
          top_margin=5mm)

    display(p0)
end


# Fine sweep

In [ ]:
ϵ_0_list = vcat(range(0.44,0.64,12));

t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(sqrt(2)*ω_p)*2
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,25)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            a_rot = a.*exp.(1im*ω_a*t_list)

            Final_mode[(re_i,im_j)] = a_rot
        end
    end
    Final_all[ϵ_0] = Final_mode
end

In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(-15,15),
          ylims=(-6,6),
          top_margin=5mm)

    display(p0)
end


In [ ]:
using JSON
data_lyapunov = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/ouput/data_lyapunov.json")
n_rows = 20
n_cols = 30

Im_a_list_lyap=range(-4,4,n_rows)
Re_a_list_lyap=range(-5,5,n_cols)
Lyap_coef_dict = Dict(parse(Float64,k) => reshape(v, n_rows, n_cols) for (k,v) in data_lyapunov)
Lyap_coef_clean = Dict(
    k => replace(v, nothing => NaN)
    for (k, v) in Lyap_coef_dict
)

Lyap_coef_sign = Dict(k => map(x -> x>0 ? 1 : -1, v) for (k, v) in Lyap_coef_clean);

In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Lyap_coef = Lyap_coef_sign[g2/2pi*1e3]
    Final_mode = Final_all[ϵ_0]

    # Plot heatmap first
    p0 = heatmap(Re_a_list_lyap, Im_a_list_lyap, Lyap_coef,
                 color=:bluesreds, colorbar=false)

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(-15,15),
          ylims=(-6,6),
          top_margin=5mm)

    display(p0)
end